# EONET API → SQLite  
## Notebook z Projektem bazy danych oraz importem bazy danych


<div style="text-align: center;">
    <img src="Onet.svg" width="900"  alt="Diagram bazy danych" />
</div>

## Skrypt tworzący baze danych

In [ ]:

DROP TABLE IF EXISTS import_log;
DROP TABLE IF EXISTS event_category;
DROP TABLE IF EXISTS event_source;
DROP TABLE IF EXISTS event_geometry;
DROP TABLE IF EXISTS source;
DROP TABLE IF EXISTS category;
DROP TABLE IF EXISTS geometry;
DROP TABLE IF EXISTS geometry_type;
DROP TABLE IF EXISTS event_location;
DROP TABLE IF EXISTS event;

CREATE TABLE event (
	id INTEGER PRIMARY KEY, -- Przychodzi z API jako EONET_x zapisujemy tylko x
	title  VARCHAR(255) NOT NULL, -- Do przecinka z tego co przydchodzi w polu title
	description TEXT,
	start_date TIMESTAMP NOT NULL,
	end_date TIMESTAMP
);

CREATE TABLE event_location (
    id SERIAL PRIMARY KEY,
    event_id INTEGER NOT NULL, 
    country_code CHAR(2),
    country VARCHAR(100),
    state VARCHAR(100),
    city VARCHAR(100),

   
    CONSTRAINT fk_event_location_event
        FOREIGN KEY (event_id) 
        REFERENCES event(id)
		ON UPDATE CASCADE 
        ON DELETE CASCADE
);

CREATE TABLE geometry_type (
	id SERIAL PRIMARY KEY,
	type VARCHAR(100) NOT NULL
);

CREATE TABLE geometry (
	id SERIAL PRIMARY KEY, -- Własny z sql
	latitude FLOAT NOT NULL CHECK (90 >= latitude AND latitude >= -90), --latitude i longitude są w takich zakresach
	longitude FLOAT NOT NULL CHECK (180 >= longitude AND longitude >= -180),
	type_id INTEGER NOT NULL,
	magnitude FLOAT,
	unit FLOAT,
	measure_date TIMESTAMP NOT NULL,

	CONSTRAINT fk_geometry_type_geometry
		FOREIGN KEY (type_id)
		REFERENCES geometry_type(id)
		ON UPDATE CASCADE
		ON DELETE RESTRICT --nie chcemy usunąć typu punktu który jest zapisany w bazie
);

CREATE TABLE category (
	id SERIAL PRIMARY KEY,
	abbreviation VARCHAR(10) NOT NULL,
	title VARCHAR(255) NOT NULL,
	description TEXT
);

CREATE TABLE source (
	id SERIAL PRIMARY KEY,
	title VARCHAR(255) NOT NULL,
	description TEXT
	
);

CREATE TABLE event_category (
	event_id INTEGER NOT NULL,
	category_id INTEGER NOT NULL,

	CONSTRAINT pk_event_category
		PRIMARY KEY (event_id, category_id),
		
	CONSTRAINT fk_event_category_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_category_category
		FOREIGN KEY (category_id)
		REFERENCES category(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_source (
	event_id INTEGER  NOT NULL,
	source_id INTEGER NOT  NULL,

	
	CONSTRAINT pk_event_source
		PRIMARY KEY (event_id, source_id),

	CONSTRAINT fk_event_source_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_source_source
		FOREIGN KEY (source_id)
		REFERENCES source(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE
);

CREATE TABLE event_geometry (
	event_id INTEGER NOT NULL,
	geometry_id INTEGER NOT NULL,


	CONSTRAINT pk_event_geometry
		PRIMARY KEY (event_id, geometry_id),

	CONSTRAINT fk_event_geometry_event
		FOREIGN KEY (event_id)
		REFERENCES event(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE,

	CONSTRAINT fk_event_geometry
		FOREIGN KEY (geometry_id)
		REFERENCES geometry(id)
		ON UPDATE CASCADE
		ON DELETE CASCADE

);

CREATE TABLE import_log (
    id SERIAL PRIMARY KEY,
    imported_at TIMESTAMP NOT NULL,
    apiname VARCHAR(255) NOT NULL,
    endpoint VARCHAR(255) NOT NULL,
    records_received INTEGER NOT NULL,
    records_saved INTEGER NOT NULL,
    status VARCHAR(100) NOT NULL,
    error_message TEXT NOT NULL
);

## Import bibliotek

In [ ]:
from __future__ import annotations



from sqlalchemy import URL, create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import pandas as pd
import reverse_geocode as rg
import requests

## Tworzenie schematu bazy danych

##### Struktura pliku konfiguracyjnego (conf.txt):
DB_USER=moj_uzytkownik\
DB_PASSWORD=moje_super_tajne_haslo\
DB_HOST=localhost\
DB_PORT=5432\
DB_NAME=moja_baza_danych

In [ ]:


config = {}
with open("conf.txt", "r", encoding="utf-8") as f:
    for linia in f:
        linia = linia.strip()
        if not linia or linia.startswith("#"):
            continue
        klucz, wartosc = linia.split("=", 1)
        config[klucz.strip()] = wartosc.strip()


db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=config.get("DB_USER"),
    password=config.get("DB_PASSWORD"),
    host=config.get("DB_HOST"),
    port=config.get("DB_PORT"),      
    database=config.get("DB_NAME"),
)


from scripts import schema



engine = create_engine(db_url)

try:
    conn = engine.connect()
    print("[INFO] POŁACZONO DO BAZY DANYCH")
except SQLAlchemyError as e:
    print(f"[ERROR] BŁĄD POŁĄCZENIA DO BAZY DANYCH {e}")

res = conn.execute(text(schema))
conn.commit()
print("[INFO] UTWORZONO BAZĘ DANYCH")
conn.close()

## Pobieranie danych z API


In [ ]:
from typing import Any
from time import sleep

import pandas as pd
import requests

API_URL_EVENTS = "https://eonet.gsfc.nasa.gov/api/v3/events"

TIMEOUT_SECONDS = 30
EVENT_LIMIT = 100
DAYS_BACK = 90

USA_BBOX = "-124.85,49.38,-66.89,24.40"


def fetch_events_from_usa() -> list[dict[str, Any]]:
    params = {
        "bbox": USA_BBOX,
        "status": "all",
        "days": DAYS_BACK,
        "limit": EVENT_LIMIT,
    }

    for attempt in range(1, 4):
        try:
            response = requests.get(
                API_URL_EVENTS,
                params=params,
                headers={"Accept": "application/json"},
                timeout=TIMEOUT_SECONDS,
            )

            response.raise_for_status()

            payload = response.json()
            events = payload.get("events", [])

            if not isinstance(events, list):
                raise ValueError("API zwróciło dane w nieoczekiwanym formacie.")

            return events

        except requests.RequestException as error:
            print(f"Próba {attempt}/3 nie powiodła się: {error}")

            if attempt < 3:
                sleep(3)

    return []


events = fetch_events_from_usa()

print(f"Liczba pobranych zdarzeń z USA: {len(events)}")

events_df = pd.DataFrame(
    [
        {
            "api_id": event.get("id"),
            "title": event.get("title"),
            "description": event.get("description"),
            "closed": event.get("closed"),
            "categories_count": len(event.get("categories", [])),
            "sources_count": len(event.get("sources", [])),
            "geometries_count": len(event.get("geometry", [])),
        }
        for event in events
    ]
)

display(events_df)


## Zapisywanie danych do tabel

## Uruchomienie pełnego importu

## Weryfikacja zawartości bazy danych

## Analiza Danych i Wizualizacja